# Advanced Python `__new__` Problems — With Full Solutions

This notebook is an advanced, executable problem set focused on Python's `__new__` method.

## Learning goals

By the end, you should be able to:

- explain the exact relationship between `__new__` and `__init__`
- predict when `__init__` will and will not execute
- subclass immutable built-in types correctly
- preserve inheritance by using cooperative `super().__new__`
- implement instance caching / interning safely
- recognize repeated-initialization bugs caused by cached instances
- reason about `cls`, subclasses, and covariance of construction
- design validated immutable value objects
- debug broken `__new__` implementations
- understand interactions with `__slots__`, pickling, weak references, and metaclasses

All examples use only Python's standard library.

## Core mental model

For a normal class call such as:

```python
obj = MyClass(a, b)
```

Python conceptually does something like:

```python
obj = MyClass.__new__(MyClass, a, b)

if isinstance(obj, MyClass):
    MyClass.__init__(obj, a, b)
```

That is a simplified model, but it captures the most important rule for this notebook:

> `__new__` creates/returns an object. `__init__` initializes that object only when the returned object is an instance of the requested class (or a subclass of it).

For immutable built-ins such as `int`, `str`, and `tuple`, value construction must generally happen in `__new__`, because by the time `__init__` runs, the immutable value already exists.

## Warm-up example: trace the constructor pipeline

In [1]:
class Trace:
    def __new__(cls, value):
        print(f"__new__ received cls={cls.__name__}, value={value!r}")
        instance = super().__new__(cls)
        print(f"__new__ created id={id(instance)}")
        return instance

    def __init__(self, value):
        print(f"__init__ received id={id(self)}, value={value!r}")
        self.value = value

t = Trace(10)
print(t.__dict__)

__new__ received cls=Trace, value=10
__new__ created id=2142330030128
__init__ received id=2142330030128, value=10
{'value': 10}


# Problem 1 — Predict the exact call order

Without running the code first, predict the output.

Then run it and explain **why** `B.__new__` delegates into `A.__new__`, and why both `__init__` methods run.

In [2]:
class A:
    def __new__(cls, x):
        print("A.__new__", cls.__name__, x)
        return super().__new__(cls)

    def __init__(self, x):
        print("A.__init__", type(self).__name__, x)

class B(A):
    def __new__(cls, x):
        print("B.__new__", cls.__name__, x)
        return super().__new__(cls, x)

    def __init__(self, x):
        print("B.__init__", type(self).__name__, x)
        super().__init__(x)

b = B(5)

B.__new__ B 5
A.__new__ B 5
B.__init__ B 5
A.__init__ B 5


### Solution 1

Expected order:

```text
B.__new__ B 5
A.__new__ B 5
B.__init__ B 5
A.__init__ B 5
```

Important detail: in `A.__new__`, `cls` is still `B`, not `A`. Cooperative construction preserves the most-derived class through the chain.

# Problem 2 — Returning the wrong type

Implement `MaybeNumber` so that:

- `MaybeNumber("123")` returns the integer `123`
- `MaybeNumber("abc")` returns an actual `MaybeNumber` instance
- `__init__` should run only for the second case

Then test both branches.

In [3]:
class MaybeNumber:
    def __new__(cls, text):
        if isinstance(text, str) and text.isdigit():
            return int(text)
        return super().__new__(cls)

    def __init__(self, text):
        print("MaybeNumber.__init__ ran")
        self.text = text

a = MaybeNumber("123")
b = MaybeNumber("abc")

print(a, type(a))
print(b, type(b), b.__dict__)

assert a == 123 and type(a) is int
assert isinstance(b, MaybeNumber)
assert b.text == "abc"

MaybeNumber.__init__ ran
123 <class 'int'>
<__main__.MaybeNumber object at 0x000001F2DCCDE900> <class '__main__.MaybeNumber'> {'text': 'abc'}


### Solution 2

When `__new__` returns an `int`, Python does not call `MaybeNumber.__init__`, because the result is not an instance of `MaybeNumber`.

When `__new__` returns an instance created through `super().__new__(cls)`, initialization continues normally.

# Problem 3 — Fix a broken immutable subclass

The following idea is wrong:

```python
class PositiveInt(int):
    def __init__(self, value):
        if value < 0:
            value = -value
```

Why does it fail to change the integer's value?

Implement the correct version using `__new__`.

In [4]:
class PositiveInt(int):
    def __new__(cls, value):
        value = int(value)
        return super().__new__(cls, abs(value))

for x in [-7, 0, 12]:
    y = PositiveInt(x)
    print(repr(y), type(y).__name__)

assert PositiveInt(-7) == 7
assert isinstance(PositiveInt(-7), int)
assert isinstance(PositiveInt(-7), PositiveInt)

7 PositiveInt
0 PositiveInt
12 PositiveInt


### Solution 3

`int` is immutable. Its numeric value must be selected while the object is being created. `__init__` runs too late to replace the stored integer value.

# Problem 4 — A normalized string type

Create a `Username` class that:

- subclasses `str`
- strips leading/trailing whitespace
- lowercases input
- replaces internal spaces with `_`
- rejects empty usernames
- remains a genuine `str`

Examples:

```python
Username("  Ada Lovelace  ") == "ada_lovelace"
Username("   ")  # raises ValueError
```

In [5]:
class Username(str):
    def __new__(cls, value):
        normalized = "_".join(str(value).strip().lower().split())
        if not normalized:
            raise ValueError("username cannot be empty")
        return super().__new__(cls, normalized)

u = Username("  Ada   Lovelace  ")
print(u, type(u))
assert u == "ada_lovelace"
assert isinstance(u, str)
assert isinstance(u, Username)

try:
    Username("   ")
except ValueError as e:
    print("Expected:", e)

ada_lovelace <class '__main__.Username'>
Expected: username cannot be empty


# Problem 5 — Tuple subclass with validation

Create an immutable `Point3D(tuple)` whose constructor is:

```python
Point3D(x, y, z)
```

Requirements:

- store exactly three floats
- expose `.x`, `.y`, `.z` properties
- reject non-finite values (`nan`, `inf`, `-inf`)
- keep tuple behavior

In [6]:
import math

class Point3D(tuple):
    def __new__(cls, x, y, z):
        values = tuple(float(v) for v in (x, y, z))
        if not all(math.isfinite(v) for v in values):
            raise ValueError("coordinates must be finite")
        return super().__new__(cls, values)

    @property
    def x(self):
        return self[0]

    @property
    def y(self):
        return self[1]

    @property
    def z(self):
        return self[2]

p = Point3D(1, 2.5, 3)
print(p, p.x, p.y, p.z)
assert p == (1.0, 2.5, 3.0)
assert isinstance(p, tuple)

try:
    Point3D(1, float("inf"), 3)
except ValueError as e:
    print("Expected:", e)

(1.0, 2.5, 3.0) 1.0 2.5 3.0
Expected: coordinates must be finite


# Problem 6 — Find the inheritance bug

Why is this constructor hierarchy fragile?

```python
class Base:
    def __new__(cls, value):
        return object.__new__(cls)

class Child(Base):
    def __new__(cls, value):
        return object.__new__(cls)
```

Rewrite it cooperatively so future classes in the hierarchy can participate.

In [7]:
class Base:
    def __new__(cls, value):
        print("Base participates")
        return super().__new__(cls)

class Child(Base):
    def __new__(cls, value):
        print("Child participates")
        return super().__new__(cls, value)

c = Child(10)
assert isinstance(c, Child)

Child participates
Base participates


### Solution 6

Calling `object.__new__` directly skips any intermediate implementation that could exist later in the MRO. `super().__new__` composes better with inheritance.

The key best practice is not "always call `super()` mechanically"; it is:

> Delegate to the next constructor in the MRO when you want cooperative inheritance.

# Problem 7 — Constructor covariance with subclasses

Implement a base class `Record` whose `__new__` stores a creation counter on each concrete subclass.

Creating `AdminRecord(...)` must produce an `AdminRecord`, not a plain `Record`.

In [8]:
class Record:
    def __new__(cls, *args, **kwargs):
        instance = super().__new__(cls)
        cls.created = getattr(cls, "created", 0) + 1
        return instance

class AdminRecord(Record):
    pass

r = Record()
a1 = AdminRecord()
a2 = AdminRecord()

print(type(r).__name__, type(a1).__name__, type(a2).__name__)
print("Record.created =", Record.created)
print("AdminRecord.created =", AdminRecord.created)

assert type(a1) is AdminRecord
assert AdminRecord.created == 3  # inherited value 1, then incremented twice

Record AdminRecord AdminRecord
Record.created = 1
AdminRecord.created = 3


### Discussion 7

This example intentionally shows a subtle class-attribute effect: `getattr(cls, "created", 0)` can see an inherited value.

If you need an independent counter per concrete class, inspect `cls.__dict__` instead.

In [9]:
class IndependentRecord:
    def __new__(cls, *args, **kwargs):
        instance = super().__new__(cls)
        current = cls.__dict__.get("created", 0)
        cls.created = current + 1
        return instance

class AdminIndependent(IndependentRecord):
    pass

IndependentRecord()
AdminIndependent()
AdminIndependent()

print(IndependentRecord.created, AdminIndependent.created)
assert IndependentRecord.created == 1
assert AdminIndependent.created == 2

1 2


# Problem 8 — Implement a classic singleton

Implement a singleton in `__new__`.

Then answer:

1. Why can `__init__` still run multiple times?
2. Why is this pattern often inferior to a module-level object or dependency injection?

In [10]:
class Singleton:
    _instance = None

    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

a = Singleton()
b = Singleton()

print(a is b)
assert a is b

True


### Solution 8

Caching an instance in `__new__` controls **allocation/identity**, but every class call can still trigger `__init__` on the returned instance.

That leads directly to the next problem.

# Problem 9 — The repeated-`__init__` singleton bug

Observe the bug, then repair it.

In [11]:
class BrokenConfig:
    _instance = None

    def __new__(cls, value):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance

    def __init__(self, value):
        self.value = value

x = BrokenConfig("first")
y = BrokenConfig("second")

print(x is y, x.value, y.value)
assert x is y
assert x.value == "second"  # surprising mutation

True second second


### One repair

Make initialization idempotent or guard it explicitly.

In [12]:
class Config:
    _instance = None

    def __new__(cls, value):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance._initialized = False
        return cls._instance

    def __init__(self, value):
        if self._initialized:
            return
        self.value = value
        self._initialized = True

x = Config("first")
y = Config("second")

print(x is y, x.value, y.value)
assert x is y
assert x.value == "first"

True first first


# Problem 10 — Flyweight / interning cache

Implement `Symbol(name)` such that repeated construction with equivalent names returns the same object.

Requirements:

- normalize to lowercase
- trim whitespace
- cache by normalized name
- use a weak-reference cache so unused objects can be garbage-collected
- prevent repeated initialization from corrupting an existing object

In [13]:
import weakref

class Symbol:
    _cache = weakref.WeakValueDictionary()

    def __new__(cls, name):
        key = str(name).strip().lower()
        if not key:
            raise ValueError("symbol name cannot be empty")

        existing = cls._cache.get(key)
        if existing is not None:
            return existing

        instance = super().__new__(cls)
        instance._symbol_key = key
        instance._initialized = False
        cls._cache[key] = instance
        return instance

    def __init__(self, name):
        if self._initialized:
            return
        self.name = self._symbol_key
        self._initialized = True

    def __repr__(self):
        return f"Symbol({self.name!r})"

a = Symbol("  Alpha ")
b = Symbol("alpha")
c = Symbol("BETA")

print(a, b, c)
print("a is b:", a is b)

assert a is b
assert a is not c
assert a.name == "alpha"

Symbol('alpha') Symbol('alpha') Symbol('beta')
a is b: True


# Problem 11 — Cache correctly across subclasses

The cache in a base class can accidentally make subclasses share instances.

Design `Interned` so cache keys include the concrete class.

In [14]:
import weakref

class Interned:
    _cache = weakref.WeakValueDictionary()

    def __new__(cls, key):
        normalized = str(key).strip().lower()
        cache_key = (cls, normalized)

        obj = cls._cache.get(cache_key)
        if obj is None:
            obj = super().__new__(cls)
            obj.key = normalized
            cls._cache[cache_key] = obj
        return obj

class RedInterned(Interned):
    pass

class BlueInterned(Interned):
    pass

r1 = RedInterned("x")
r2 = RedInterned(" X ")
b1 = BlueInterned("x")

print(r1 is r2, r1 is b1)
assert r1 is r2
assert r1 is not b1

True False


# Problem 12 — `__slots__` and `__new__`

Create a class with no instance `__dict__`.

Use `__new__` to precompute a normalized value and `__init__` to store the original input.

In [15]:
class SlottedName:
    __slots__ = ("original", "normalized")

    def __new__(cls, value):
        obj = super().__new__(cls)
        obj.normalized = str(value).strip().casefold()
        return obj

    def __init__(self, value):
        self.original = str(value)

s = SlottedName("  PyThOn  ")
print(s.original, s.normalized)
print("has __dict__?", hasattr(s, "__dict__"))

assert s.normalized == "python"
assert not hasattr(s, "__dict__")

  PyThOn   python
has __dict__? False


# Problem 13 — Validation before allocation

Create `BoundedInt(int)` with constructor:

```python
BoundedInt(value, minimum=0, maximum=100)
```

Rules:

- coerce `value` with `int(value)`
- require `minimum <= value <= maximum`
- preserve the actual subclass
- attach `minimum` and `maximum` metadata if possible

Then explain the metadata limitation of some immutable built-ins.

In [16]:
class BoundedInt(int):
    def __new__(cls, value, minimum=0, maximum=100):
        value = int(value)
        minimum = int(minimum)
        maximum = int(maximum)

        if minimum > maximum:
            raise ValueError("minimum cannot exceed maximum")
        if not minimum <= value <= maximum:
            raise ValueError(f"{value} is outside [{minimum}, {maximum}]")

        obj = super().__new__(cls, value)
        obj.minimum = minimum
        obj.maximum = maximum
        return obj

x = BoundedInt("42", 10, 50)
print(x, x.minimum, x.maximum, type(x))
assert x == 42
assert x.minimum == 10

42 10 50 <class '__main__.BoundedInt'>


# Problem 14 — An immutable money value

Implement `Money(tuple)` representing `(currency, cents)`.

Requirements:

- normalize currency to uppercase
- currency must be exactly 3 alphabetic characters
- cents must be an integer
- expose `.currency` and `.cents`
- implement addition only for matching currencies

In [17]:
class Money(tuple):
    def __new__(cls, currency, cents):
        currency = str(currency).strip().upper()
        if len(currency) != 3 or not currency.isalpha():
            raise ValueError("currency must be a 3-letter code")

        if isinstance(cents, bool):
            raise TypeError("cents must be an integer, not bool")

        cents = int(cents)
        return super().__new__(cls, (currency, cents))

    @property
    def currency(self):
        return self[0]

    @property
    def cents(self):
        return self[1]

    def __add__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        if self.currency != other.currency:
            raise ValueError("cannot add different currencies")
        return type(self)(self.currency, self.cents + other.cents)

m1 = Money("usd", 125)
m2 = Money("USD", 75)
m3 = m1 + m2

print(m3, type(m3))
assert m3 == ("USD", 200)

('USD', 200) <class '__main__.Money'>


# Problem 15 — Preserve subclasses in immutable operations

Subclass `Money` as `TaxedMoney`.

Check whether `m1 + m2` preserves the concrete subclass and explain why `type(self)(...)` is preferable to hard-coding `Money(...)`.

In [18]:
class TaxedMoney(Money):
    pass

t1 = TaxedMoney("eur", 100)
t2 = TaxedMoney("eur", 50)
t3 = t1 + t2

print(t3, type(t3))
assert isinstance(t3, TaxedMoney)
assert t3.cents == 150

('EUR', 150) <class '__main__.TaxedMoney'>


# Problem 16 — A constructor that sometimes reuses an object

Implement `CanonicalPath(str)`:

- collapse repeated `/`
- remove trailing `/` except for root
- return `/` for any path consisting only of slashes
- intern identical normalized paths
- remain a `str`

In [19]:
class CanonicalPath(str):
    _cache = {}

    def __new__(cls, path):
        parts = [part for part in str(path).split("/") if part]

        if str(path).startswith("/"):
            normalized = "/" + "/".join(parts)
        else:
            normalized = "/".join(parts)

        if normalized == "":
            normalized = "/" if "/" in str(path) else "."

        if normalized != "/":
            normalized = normalized.rstrip("/")

        key = (cls, normalized)
        if key in cls._cache:
            return cls._cache[key]

        obj = super().__new__(cls, normalized)
        cls._cache[key] = obj
        return obj

p1 = CanonicalPath("/a//b///")
p2 = CanonicalPath("/a/b")
root = CanonicalPath("////")

print(p1, p1 is p2, root)
assert p1 == "/a/b"
assert p1 is p2
assert root == "/"

/a/b True /


# Problem 17 — Debugging: forgetting to return the instance

What happens when `__new__` forgets to return?

In [20]:
class Broken:
    def __new__(cls):
        super().__new__(cls)
        # implicit return None

    def __init__(self):
        print("This will not run.")

result = Broken()

print(result, type(result))
assert result is None

None <class 'NoneType'>


### Solution 17

A Python function with no explicit return statement returns `None`.

Therefore the class call returns `None`, and `__init__` is skipped because `None` is not an instance of `Broken`.

# Problem 18 — Debugging: incompatible `__new__` and `__init__` signatures

The constructor pipeline forwards the class-call arguments.

Repair this class so both methods accept compatible arguments.

In [21]:
class Pair:
    def __new__(cls, x, y):
        obj = super().__new__(cls)
        return obj

    def __init__(self, x, y):
        self.x = x
        self.y = y

pair = Pair(2, 3)
print(pair.__dict__)
assert (pair.x, pair.y) == (2, 3)

{'x': 2, 'y': 3}


# Problem 19 — Keyword-only constructor API

Build an immutable `Temperature(float)` that can only be called like:

```python
Temperature(celsius=36.5)
```

Also provide `.fahrenheit`.

In [22]:
class Temperature(float):
    def __new__(cls, *, celsius):
        value = float(celsius)
        if value < -273.15:
            raise ValueError("temperature below absolute zero")
        return super().__new__(cls, value)

    @property
    def celsius(self):
        return float(self)

    @property
    def fahrenheit(self):
        return self * 9 / 5 + 32

t = Temperature(celsius=36.5)
print(t.celsius, t.fahrenheit)
assert round(t.fahrenheit, 1) == 97.7

try:
    Temperature(36.5)
except TypeError as e:
    print("Expected:", e)

36.5 97.7
Expected: Temperature.__new__() takes 1 positional argument but 2 were given


# Problem 20 — `__new__` as a class-selection factory

A constructor can return an instance of another, unrelated class.

Implement `Shape(kind, **kwargs)` so:

```python
Shape("circle", radius=2)
Shape("rectangle", width=3, height=4)
```

returns concrete `Circle` or `Rectangle` objects.

Here `Circle` and `Rectangle` deliberately do **not** inherit from `Shape`. Therefore the object returned by `Shape.__new__` is not an instance of `Shape`, and `Shape` initialization is skipped. This makes the constructor-selection behavior explicit for the purpose of the exercise.

A classmethod factory is usually clearer for this design; the next problem shows that alternative.


In [23]:
import math

class Shape:
    def __new__(cls, kind, **kwargs):
        kind = kind.lower()

        if kind == "circle":
            return Circle(**kwargs)
        if kind == "rectangle":
            return Rectangle(**kwargs)

        raise ValueError(f"unknown shape: {kind}")

class Circle:
    def __init__(self, radius):
        self.radius = float(radius)

    @property
    def area(self):
        return math.pi * self.radius ** 2

class Rectangle:
    def __init__(self, width, height):
        self.width = float(width)
        self.height = float(height)

    @property
    def area(self):
        return self.width * self.height

c = Shape("circle", radius=2)
r = Shape("rectangle", width=3, height=4)

print(type(c).__name__, c.area)
print(type(r).__name__, r.area)

assert isinstance(c, Circle)
assert isinstance(r, Rectangle)
assert not isinstance(c, Shape)
assert r.area == 12


Circle 12.566370614359172
Rectangle 12.0


# Problem 21 — Safer factory alternative

The previous pattern works, but a classmethod factory is often clearer because object selection is explicit.

Rewrite it using `Shape.create(...)`.

In [24]:
class ShapeFactoryBase:
    @classmethod
    def create(cls, kind, **kwargs):
        kind = kind.lower()
        if kind == "circle":
            return Circle2(**kwargs)
        if kind == "rectangle":
            return Rectangle2(**kwargs)
        raise ValueError(kind)

class Circle2(ShapeFactoryBase):
    def __init__(self, radius):
        self.radius = float(radius)

class Rectangle2(ShapeFactoryBase):
    def __init__(self, width, height):
        self.width = float(width)
        self.height = float(height)

obj = ShapeFactoryBase.create("circle", radius=5)
print(type(obj).__name__, obj.radius)
assert isinstance(obj, Circle2)

Circle2 5.0


# Problem 22 — `__new__` plus `__slots__` plus weak references

Weak-value caches require objects to support weak references.

Create a slotted, interned `Token` class that supports weak references.

In [25]:
import weakref

class Token:
    __slots__ = ("text", "__weakref__")
    _cache = weakref.WeakValueDictionary()

    def __new__(cls, text):
        key = str(text).strip()
        obj = cls._cache.get(key)

        if obj is None:
            obj = super().__new__(cls)
            obj.text = key
            cls._cache[key] = obj

        return obj

a = Token("abc")
b = Token("abc")

print(a is b, a.text)
assert a is b

True abc


# Problem 23 — Identity cache with an expensive payload

Implement `Dataset(name)` where loading happens only once per normalized dataset name.

Simulate loading with a counter rather than I/O.

In [26]:
class Dataset:
    _cache = {}
    loads = 0

    def __new__(cls, name):
        key = str(name).strip().casefold()

        if key in cls._cache:
            return cls._cache[key]

        obj = super().__new__(cls)
        obj._initialized = False
        obj._key = key
        cls._cache[key] = obj
        return obj

    def __init__(self, name):
        if self._initialized:
            return

        type(self).loads += 1
        self.name = self._key
        self.payload = f"loaded:{self.name}"
        self._initialized = True

d1 = Dataset(" Sales ")
d2 = Dataset("sales")

print(d1 is d2, Dataset.loads, d1.payload)
assert d1 is d2
assert Dataset.loads == 1

True 1 loaded:sales


# Problem 24 — Pickling an immutable subclass

Immutable subclasses with custom constructor signatures can require explicit pickle support.

Implement `TaggedInt(int)` and ensure a pickle round trip preserves both:

- integer value
- `tag`

In [27]:
import pickle

class TaggedInt(int):
    def __new__(cls, value, tag):
        obj = super().__new__(cls, int(value))
        obj.tag = str(tag)
        return obj

    def __getnewargs_ex__(self):
        return ((int(self), self.tag), {})

x = TaggedInt(7, "lucky")
blob = pickle.dumps(x)
y = pickle.loads(blob)

print(y, y.tag, type(y))
assert y == 7
assert y.tag == "lucky"
assert isinstance(y, TaggedInt)

7 lucky <class '__main__.TaggedInt'>


# Problem 25 — Make an immutable validated identifier

Implement `UserId(int)` with all of these constraints:

- accepted input: `int` or decimal string
- booleans are rejected
- range is `1 <= id <= 2**31 - 1`
- subclass identity is preserved
- no mutation-based validation in `__init__`

In [28]:
class UserId(int):
    def __new__(cls, value):
        if isinstance(value, bool):
            raise TypeError("bool is not a valid user id")

        if isinstance(value, str):
            text = value.strip()
            if not text.isdecimal():
                raise ValueError("string user id must contain only decimal digits")
            value = int(text)
        elif isinstance(value, int):
            value = int(value)
        else:
            raise TypeError("user id must be int or decimal string")

        if not (1 <= value <= 2**31 - 1):
            raise ValueError("user id out of range")

        return super().__new__(cls, value)

ids = [UserId(1), UserId("42"), UserId(2**31 - 1)]
print(ids)

for bad in [True, "12x", 0, 2**31]:
    try:
        UserId(bad)
    except (TypeError, ValueError) as e:
        print(f"{bad!r}: {type(e).__name__}: {e}")

[1, 42, 2147483647]
True: TypeError: bool is not a valid user id
'12x': ValueError: string user id must contain only decimal digits
0: ValueError: user id out of range
2147483648: ValueError: user id out of range


# Problem 26 — Multiple inheritance and cooperative `__new__`

Use mixins whose `__new__` methods all participate without hard-coding each other.

In [29]:
class LoggingNewMixin:
    def __new__(cls, *args, **kwargs):
        print("LoggingNewMixin")
        return super().__new__(cls)

class AuditNewMixin:
    def __new__(cls, *args, **kwargs):
        print("AuditNewMixin")
        obj = super().__new__(cls)
        obj.audit_created = True
        return obj

class Entity(LoggingNewMixin, AuditNewMixin):
    def __init__(self, name):
        self.name = name

print(Entity.__mro__)
e = Entity("alpha")

print(e.name, e.audit_created)
assert e.audit_created is True

(<class '__main__.Entity'>, <class '__main__.LoggingNewMixin'>, <class '__main__.AuditNewMixin'>, <class 'object'>)
LoggingNewMixin
AuditNewMixin
alpha True


### Solution 26

Both mixins call `super().__new__(cls)`, so Python's MRO determines the next participant.

This is the constructor equivalent of cooperative multiple inheritance.

# Problem 27 — Why passing arbitrary args to `object.__new__` is wrong

Modern Python expects `object.__new__` to receive only the class argument.

Demonstrate the failure safely.

In [30]:
class Plain:
    pass

try:
    object.__new__(Plain, 1, 2)
except TypeError as e:
    print("Expected TypeError:", e)

Expected TypeError: Plain() takes no arguments


# Problem 28 — Build a compact immutable semantic version

Implement `Version(tuple)` supporting:

```python
Version(1, 4, 2)
Version("1.4.2")
```

Requirements:

- three non-negative integers
- string form: `"1.4.2"`
- tuple comparison should work naturally

In [31]:
class Version(tuple):
    def __new__(cls, *args):
        if len(args) == 1 and isinstance(args[0], str):
            parts = args[0].strip().split(".")
        else:
            parts = args

        if len(parts) != 3:
            raise ValueError("version must have exactly 3 components")

        try:
            values = tuple(int(p) for p in parts)
        except (TypeError, ValueError):
            raise ValueError("version components must be integers") from None

        if any(v < 0 for v in values):
            raise ValueError("version components must be non-negative")

        return super().__new__(cls, values)

    def __str__(self):
        return ".".join(map(str, self))

v1 = Version("1.4.2")
v2 = Version(1, 5, 0)

print(v1, v2, v1 < v2)
assert v1 < v2
assert tuple(v1) == (1, 4, 2)

1.4.2 1.5.0 True


# Problem 29 — Challenge: immutable RGB color

Create `Color(tuple)` that accepts either:

```python
Color(255, 128, 0)
Color("#ff8000")
```

Requirements:

- channels are integers from 0 to 255
- expose `.r`, `.g`, `.b`
- `.hex` returns uppercase hex
- instances are hashable

In [32]:
class Color(tuple):
    def __new__(cls, *args):
        if len(args) == 1 and isinstance(args[0], str):
            s = args[0].strip()
            if len(s) != 7 or not s.startswith("#"):
                raise ValueError("hex color must look like #RRGGBB")
            try:
                channels = tuple(int(s[i:i+2], 16) for i in (1, 3, 5))
            except ValueError:
                raise ValueError("invalid hex color") from None
        elif len(args) == 3:
            if any(isinstance(x, bool) for x in args):
                raise TypeError("color channels cannot be bool")
            channels = tuple(int(x) for x in args)
        else:
            raise TypeError("Color expects (r, g, b) or '#RRGGBB'")

        if not all(0 <= c <= 255 for c in channels):
            raise ValueError("channels must be in range 0..255")

        return super().__new__(cls, channels)

    r = property(lambda self: self[0])
    g = property(lambda self: self[1])
    b = property(lambda self: self[2])

    @property
    def hex(self):
        return f"#{self.r:02X}{self.g:02X}{self.b:02X}"

orange1 = Color(255, 128, 0)
orange2 = Color("#ff8000")

print(orange1, orange1.hex, orange1 == orange2)
assert orange1 == orange2
assert orange1.hex == "#FF8000"
assert hash(orange1) == hash(orange2)

(255, 128, 0) #FF8000 True


# Problem 30 — Challenge: canonical rational numbers

Subclass `tuple` as `Rational(numerator, denominator)`.

Requirements:

- denominator may not be zero
- reduce by greatest common divisor
- denominator must be positive
- `Rational(2, 4) == (1, 2)`
- `Rational(1, -2) == (-1, 2)`
- add two rationals and preserve subclass type

In [33]:
import math

class Rational(tuple):
    def __new__(cls, numerator, denominator=1):
        n = int(numerator)
        d = int(denominator)

        if d == 0:
            raise ZeroDivisionError("denominator cannot be zero")

        if d < 0:
            n, d = -n, -d

        g = math.gcd(n, d)
        n //= g
        d //= g

        return super().__new__(cls, (n, d))

    @property
    def numerator(self):
        return self[0]

    @property
    def denominator(self):
        return self[1]

    def __add__(self, other):
        if not isinstance(other, Rational):
            return NotImplemented

        n = self.numerator * other.denominator + other.numerator * self.denominator
        d = self.denominator * other.denominator
        return type(self)(n, d)

    def __repr__(self):
        return f"{type(self).__name__}({self.numerator}, {self.denominator})"

a = Rational(2, 4)
b = Rational(1, 6)
c = a + b

print(a, b, c)
assert a == (1, 2)
assert Rational(1, -2) == (-1, 2)
assert c == (2, 3)

Rational(1, 2) Rational(1, 6) Rational(2, 3)


# Problem 31 — Metaclass perspective

Class calls are ultimately handled by the metaclass's `__call__`, which normally coordinates `__new__` and `__init__`.

Instrument that process.

In [34]:
class TraceMeta(type):
    def __call__(cls, *args, **kwargs):
        print("TraceMeta.__call__: before")
        obj = super().__call__(*args, **kwargs)
        print("TraceMeta.__call__: after")
        return obj

class MetaDemo(metaclass=TraceMeta):
    def __new__(cls, value):
        print("MetaDemo.__new__")
        return super().__new__(cls)

    def __init__(self, value):
        print("MetaDemo.__init__")
        self.value = value

m = MetaDemo(99)
assert m.value == 99

TraceMeta.__call__: before
MetaDemo.__new__
MetaDemo.__init__
TraceMeta.__call__: after


# Problem 32 — Diagnose five `__new__` anti-patterns

For each pattern, state the issue.

### A
```python
def __new__(cls, x):
    obj = super().__new__(Base)
    return obj
```

### B
```python
def __new__(cls, x):
    super().__new__(cls)
```

### C
```python
def __new__(cls, x):
    return object()
```

### D
```python
class MyInt(int):
    def __init__(self, x):
        self = int(x) * 2
```

### E
```python
def __new__(cls, x):
    return cls(x)
```

## Solution 32

### A — hard-codes the wrong class
`super().__new__(Base)` can create the wrong type and breaks subclass-aware construction. Usually pass `cls`.

### B — forgets to return
The result becomes `None`, so initialization is skipped.

### C — returns an unrelated type
The class call returns a plain `object`; the class's `__init__` will not run.

### D — rebinding `self` changes only a local variable
It does not change an immutable integer object. Construct the desired integer value in `__new__`.

### E — infinite recursion
`cls(x)` calls the class again, which calls `__new__` again, and so on. Delegate to a base implementation instead.

# Problem 33 — Unit-test a `__new__` implementation

Good constructor code should be tested for:

- value
- concrete type
- identity behavior
- validation
- subclass behavior

Write compact tests for `Username`.

In [35]:
def test_username():
    u = Username("  Ada  Lovelace ")
    assert u == "ada_lovelace"
    assert type(u) is Username
    assert isinstance(u, str)

    class AdminUsername(Username):
        pass

    admin = AdminUsername(" Root ")
    assert type(admin) is AdminUsername
    assert admin == "root"

    try:
        Username("   ")
    except ValueError:
        pass
    else:
        raise AssertionError("empty username should fail")

test_username()
print("Username tests passed.")

Username tests passed.


# Problem 34 — Mini-project: immutable case-insensitive key

Build `CIKey(str)`:

- displayed value preserves the original normalized spelling
- equality and hashing should be case-insensitive
- whitespace around the key is ignored
- empty keys are rejected

Hint: subclassing `str` while changing equality/hash semantics is advanced; if you override `__eq__`, you must ensure `__hash__` remains consistent.

In [36]:
class CIKey(str):
    def __new__(cls, value):
        display = str(value).strip()
        if not display:
            raise ValueError("key cannot be empty")

        obj = super().__new__(cls, display)
        obj._folded = display.casefold()
        return obj

    def __eq__(self, other):
        if isinstance(other, str):
            return self._folded == str(other).strip().casefold()
        return NotImplemented

    def __hash__(self):
        return hash(self._folded)

a = CIKey(" Content-Type ")
b = CIKey("content-type")

print(a, b, a == b, hash(a) == hash(b))
assert a == b
assert hash(a) == hash(b)

d = {a: 123}
assert d[b] == 123

Content-Type content-type True True


# Problem 35 — Final challenge: typed immutable percentage

Implement `Percentage(float)`.

Requirements:

- `Percentage(25)` means `25%`, internally stored as `0.25`
- `Percentage(0.25, normalized=True)` also stores `0.25`
- valid interval is `[0, 1]` internally
- `.percent` returns `25.0`
- string representation is `"25.00%"`
- arithmetic with another `Percentage` is allowed only where semantically meaningful; implement multiplication by a scalar but not addition beyond 100%

In [37]:
class Percentage(float):
    def __new__(cls, value, *, normalized=False):
        raw = float(value)
        fraction = raw if normalized else raw / 100.0

        if not 0.0 <= fraction <= 1.0:
            raise ValueError("percentage must be between 0% and 100%")

        return super().__new__(cls, fraction)

    @property
    def percent(self):
        return float(self) * 100

    def __str__(self):
        return f"{self.percent:.2f}%"

    def __mul__(self, scalar):
        if isinstance(scalar, (int, float)):
            return float(self) * scalar
        return NotImplemented

    __rmul__ = __mul__

p = Percentage(25)
q = Percentage(0.25, normalized=True)

print(p, q, p * 200)
assert p == q == 0.25
assert p.percent == 25.0
assert p * 200 == 50.0

25.00% 25.00% 50.0


# Best-practice checklist

Use `__new__` when you genuinely need to control **object creation** rather than ordinary post-creation initialization.

## Strong use cases

- subclassing immutable built-ins (`int`, `str`, `tuple`, `float`, etc.)
- interning / flyweight caches
- singleton-like identity control
- validating or transforming data before immutable construction
- returning a specialized object from a construction API
- low-level framework or metaprogramming work

## Prefer `__init__` when

- you simply need to assign ordinary mutable instance attributes
- identity/allocation does not need customization

## Prefer a factory function or classmethod when

- returning multiple unrelated concrete types
- the caller should clearly see that selection/factory behavior is occurring

## Constructor hygiene

1. Usually delegate with `super().__new__(cls, ...)` when inheritance should remain cooperative.
2. Always return the intended object.
3. Remember that cached/reused objects can cause `__init__` to run repeatedly.
4. Ensure `__new__` and `__init__` accept compatible class-call arguments.
5. For immutable subclasses, validate and transform before calling the parent `__new__`.
6. Test concrete type, value, identity, subclass behavior, and failure cases.
7. Avoid recursive `cls(...)` calls from inside `__new__` unless recursion is explicitly controlled.
8. Be cautious with global caches; weak references can prevent unbounded retention.
9. If pickling matters, test round trips for immutable subclasses with custom constructor signatures.
10. Do not use `__new__` merely because it is possible—use it when allocation semantics actually require it.

# Extra drills — no solution immediately below

Try these before looking back at the earlier patterns.

### Drill A
Create `EvenInt(int)` that rounds odd inputs upward to the next even integer.

### Drill B
Create `NonEmptyTuple(tuple)` that rejects zero elements.

### Drill C
Create a weakly interned `Coordinate(x, y)` where identical coordinates share identity.

### Drill D
Create `LowerStr(str)` that canonicalizes with `.casefold()`.

### Drill E
Create a base class whose `__new__` records the concrete subclass name on each instance while remaining safe for inheritance.

### Drill F
Modify `Rational` so multiplication preserves subclasses.

### Drill G
Make `Version` accept two components and default patch to zero.

### Drill H
Design a failure test proving that `__init__` is skipped when `__new__` returns an unrelated object.

# Compact reference

```python
class MutableExample:
    def __new__(cls, *args, **kwargs):
        obj = super().__new__(cls)
        return obj

    def __init__(self, ...):
        ...
```

Immutable subclass pattern:

```python
class MyInt(int):
    def __new__(cls, value):
        transformed = ...
        return super().__new__(cls, transformed)
```

Interning pattern:

```python
class Interned:
    _cache = {}

    def __new__(cls, key):
        cache_key = (cls, normalize(key))
        if cache_key in cls._cache:
            return cls._cache[cache_key]

        obj = super().__new__(cls)
        cls._cache[cache_key] = obj
        return obj
```

When cached objects also use `__init__`, make initialization idempotent or guarded.